# 時系列データの前処理と再帰予測（リークなし）: date・Y1〜Y10・X1〜X50

- **作成者**: ImoZukushi
- **作成日**: 2026-10-21
- **目的**: `data/` 以下の学習データ（列 `date`, `Y1`〜`Y10`, `X1`〜`X50`）とテストデータ（列 `date`, `X1`〜`X50`）を
  読み込み、目的変数・日付・説明変数のそれぞれに `src/feature_engineering` の前処理をかけて CSV に出力する。
  目的変数のラグ・移動平均は、テスト期間では**再帰予測の中で**（予測値を履歴に足しながら）作る。
  目的変数は10種類あり、それぞれ別のモデルで予測する。
- **入力**（ダミー。`scripts/generate_dummy_x50.py` で作成）
  - 学習: `data/synthetic/x50/dummy_x50_train.csv`（2022-01-01〜2024-12-31）
  - テスト: `data/synthetic/x50/dummy_x50_test.csv`（2025-01-01〜2025-12-31、目的変数なし）
  - 正解: `data/synthetic/x50/dummy_x50_test_answer.csv`（予測の評価だけに使う）
- **出力**（`data/processed/x50/`）
  - `x50_train_features.csv`: 学習データの前処理結果（全目的変数のラグ・移動平均を含む）
  - `x50_test_features.csv`: テストデータの前処理結果（目的変数の特徴量は含まない）
  - `x50_test_predictions.csv`: 目的変数ごとの再帰予測の結果と、その予測に使った目的変数の特徴量

## リークを防ぐための決まり

1. **値を学習する前処理は、学習データだけで fit する。**
   べき変換の λ、スケーリングの中央値・四分位範囲、経過日数の基準日、どの列をべき変換するかの判定は、
   学習データだけで決める。テストデータには `transform` だけをかける。
2. **テストデータの目的変数は使わない。**
   テストデータには目的変数の列が無い（あれば読み込み時に外す）。正解ファイルは、最後の評価でだけ読む。
3. **目的変数のラグ・移動平均は、予測時点で分かっている値だけから作る。**
   - 学習データでは、各日 t の特徴量を t − 1 日以前の実測値から作る（当日の値を含まない）。
   - テスト期間では実測値が分からないため、特徴量を事前に作らない。再帰予測で1日ずつ進み、
     「学習期間の実測値 + それまでに予測した値」から、その日の特徴量を作ってから予測する。
4. **目的変数ごとのモデルは、自分自身の特徴量だけを使う。**
   Yk のモデルに他の目的変数（Yj）のラグを入れると、テスト期間では Yj も予測しないと値が作れない。
   そのため Yk のモデルの特徴量は「Yk のラグ・移動平均 + 日付 + 説明変数」にする。
5. **説明変数は、テスト期間の値が予測時点で分かっているとみなす**（テストデータに与えられているため）。

**前提:** 1行 = 1日で、日付に欠けや重複がないこと。テストデータは学習データの最終日の翌日から始まること。
読み込み時に確認する。

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from lightgbm import LGBMRegressor
from sklearn.pipeline import Pipeline

# pyproject.toml を目印にリポジトリルートを探し、src/ をimportパスに追加する
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

# src/ 配下のモジュール（上でパスを追加した後にimportする必要がある）
from feature_engineering.datetime_features import (  # noqa: E402
    CyclicalFeaturesEncoder,
    DatetimeFeaturesExtractor,
    ElapsedTimeTransformer,
)
from feature_engineering.numeric import LogTransformer, PandasPowerTransformer  # noqa: E402
from feature_engineering.scaling import PandasRobustScaler  # noqa: E402
from modeling.config import ForecastConfig  # noqa: E402
from modeling.forecasting import TargetFeatureBuilder, recursive_forecast  # noqa: E402
from util.csv_io import read_csv_auto  # noqa: E402
from util.paths import data_dir, ensure_parent_dir  # noqa: E402

# pandas の表示で列を省略しすぎないようにする
pd.set_option("display.max_columns", 30)


def shown(path: Path) -> str:
    """表示用のパス（リポジトリ内ならルートからの相対パス）。"""
    resolved = path.resolve()
    if resolved.is_relative_to(REPO_ROOT):
        return resolved.relative_to(REPO_ROOT).as_posix()
    return str(path)

## パラメータ

papermill で実行するときは `-p 名前 値` で上書きできる（このセルに `parameters` タグを付けている）。
リストや真偽値は `-y` で YAML として渡す。

```bash
uv run papermill notebook/003_feature_engineering_x50.ipynb out.ipynb --cwd notebook \
    -y "{TARGET_COLS: [Y1, Y2], TARGET_LAGS: [1, 7, 14], RATE_OF_CHANGE: false}"
```

In [2]:
# 入力・出力（data/ からの相対パス）。ANSWER_PATH を None にすると評価を飛ばす
TRAIN_PATH = "synthetic/x50/dummy_x50_train.csv"
TEST_PATH = "synthetic/x50/dummy_x50_test.csv"
ANSWER_PATH = "synthetic/x50/dummy_x50_test_answer.csv"
OUTPUT_DIR = "processed/x50"

# 列の指定
DATE_COL = "date"  # 日付
TARGET_COLS = [f"Y{k}" for k in range(1, 11)]  # 目的変数（それぞれ別のモデルで予測する）
FEATURE_COLS = None  # 説明変数（None なら日付・目的変数以外の全列）

# 目的変数の特徴量（全目的変数で同じ定義。再帰予測でも同じ定義で作る）
TARGET_LAGS = [1, 2, 3, 7, 14, 28]  # ラグ（日数。1以上）
ROLLING_WINDOWS = [7, 28]  # 前日までの移動平均の窓幅（日数）
RATE_OF_CHANGE = True  # 前日の前日比を加えるか
# 目的変数の変換（"log" または None）。モデルは変換後の値を学習・予測する
TARGET_TRANSFORM = "log"
TARGET_LOG_OFFSET = 0.0

# 日付から取り出す成分
DATE_COMPONENTS = ["year", "month", "day", "weekday", "week"]
# 学習データの歪度の絶対値がこれを超える説明変数は、べき変換（Yeo-Johnson）で分布を整える
SKEW_THRESHOLD = 1.0

# 再帰予測に使うモデル（LightGBM）のパラメータ（全目的変数で共通）
MODEL_PARAMS = {"n_estimators": 300, "learning_rate": 0.05, "num_leaves": 15, "random_state": 0}

In [3]:
train_path = data_dir() / TRAIN_PATH
test_path = data_dir() / TEST_PATH
answer_path = None if ANSWER_PATH is None else data_dir() / ANSWER_PATH
output_dir = data_dir() / OUTPUT_DIR
# 出力先が生データ（data/raw・data/external）でないことを確かめる（生データは書き換えない）
for immutable in ("raw", "external"):
    if output_dir.resolve().is_relative_to((data_dir() / immutable).resolve()):
        raise ValueError(f"data/{immutable} には書き込めません: {output_dir}")

# 目的変数ごとの特徴量の定義（学習データ・再帰予測の両方で同じものを使う）。
# ラグは1日以上前、移動平均・前日比は前日のラグ列から作るため、当日の値を含まない
forecast_config = ForecastConfig(
    lags=TARGET_LAGS, rolling_windows=ROLLING_WINDOWS, rate_of_change=RATE_OF_CHANGE
)
builders = {
    target: TargetFeatureBuilder(target, DATE_COL, forecast_config) for target in TARGET_COLS
}
target_feature_cols = {target: b.feature_names for target, b in builders.items()}
all_target_feature_cols = [c for cols in target_feature_cols.values() for c in cols]

print(f"学習: {shown(train_path)}")
print(f"テスト: {shown(test_path)}")
print(f"出力: {shown(output_dir)}")
print(f"目的変数: {TARGET_COLS}")
first = TARGET_COLS[0]
print(f"目的変数の特徴量（{first} の例）: {target_feature_cols[first]}")
print(f"再帰予測で必要な過去の日数: {builders[first].lookback}")

学習: data/synthetic/x50/dummy_x50_train.csv
テスト: data/synthetic/x50/dummy_x50_test.csv
出力: data/processed/x50
目的変数: ['Y1', 'Y2', 'Y3', 'Y4', 'Y5', 'Y6', 'Y7', 'Y8', 'Y9', 'Y10']
目的変数の特徴量（Y1 の例）: ['Y1_lag_1', 'Y1_lag_2', 'Y1_lag_3', 'Y1_lag_7', 'Y1_lag_14', 'Y1_lag_28', 'Y1_lag_1_ma_7', 'Y1_lag_1_ma_28', 'Y1_lag_1_roc_1']
再帰予測で必要な過去の日数: 28


## 1. 読み込みと確認

`util.csv_io.read_csv_auto` で読み込む（文字コードの判定・日付列の変換を自動で行う）。
日付の昇順に並べ、1行 = 1日か、テストデータが学習データの翌日から続いているかを確かめる。

In [4]:
def check_daily(frame: pd.DataFrame, name: str) -> None:
    """日付が1日刻みで、重複も欠けもないことを確かめる。"""
    dates = pd.to_datetime(frame[DATE_COL])
    if dates.duplicated().any():
        raise ValueError(
            f"{name}: 日付が重複しています: {dates[dates.duplicated()].head().tolist()}"
        )
    if not (dates.diff().dropna() == pd.Timedelta(days=1)).all():
        raise ValueError(f"{name}: 日付に欠けがあります（1行=1日の前提で特徴量を作るため）")


train_raw = read_csv_auto(train_path).sort_values(DATE_COL, kind="stable").reset_index(drop=True)
test_raw = read_csv_auto(test_path).sort_values(DATE_COL, kind="stable").reset_index(drop=True)
missing_targets = [t for t in TARGET_COLS if t not in train_raw.columns]
if missing_targets:
    raise ValueError(f"学習データに目的変数がありません: {missing_targets}")
leaked = [t for t in TARGET_COLS if t in test_raw.columns]
if leaked:
    # テスト期間の目的変数は予測時点で分からない値なので、あっても使わない
    print(f"テストデータの目的変数 {leaked} は使わないため外します")
    test_raw = test_raw.drop(columns=leaked)

excluded = {DATE_COL, *TARGET_COLS}
feature_cols = FEATURE_COLS or [c for c in train_raw.columns if c not in excluded]
missing_in_test = [c for c in feature_cols if c not in test_raw.columns]
if missing_in_test:
    raise ValueError(f"テストデータに説明変数がありません: {missing_in_test}")
train_raw = train_raw[[DATE_COL, *TARGET_COLS, *feature_cols]]
test_raw = test_raw[[DATE_COL, *feature_cols]]

check_daily(train_raw, "学習データ")
check_daily(test_raw, "テストデータ")
train_end, test_start = train_raw[DATE_COL].max(), test_raw[DATE_COL].min()
if test_start != train_end + pd.Timedelta(days=1):
    raise ValueError(
        f"テストデータが学習データの翌日から始まっていません: {train_end} → {test_start}"
    )

for name, frame in (("学習", train_raw), ("テスト", test_raw)):
    dates = frame[DATE_COL]
    n_missing_cols = int((frame.isna().sum() > 0).sum())
    print(
        f"{name}: {len(frame)}行 × {frame.shape[1]}列"
        f"（{dates.min():%Y-%m-%d}〜{dates.max():%Y-%m-%d}） 欠損のある列: {n_missing_cols}"
    )
# 目的変数ごとの水準・ばらつき（尺度が大きく違うので、MAE は目的変数ごとに比べる）
train_raw[TARGET_COLS].describe().T[["mean", "std", "min", "max"]].round(2)

学習: 1096行 × 61列（2022-01-01〜2024-12-31） 欠損のある列: 10
テスト: 365行 × 51列（2025-01-01〜2025-12-31） 欠損のある列: 10


,mean,std,min,max
Y1,59.83,13.34,21.32,97.40
Y2,86.37,13.86,46.13,132.19
Y3,79.47,12.33,45.05,111.54
Y4,30.20,3.70,19.39,40.90
Y5,77.85,12.48,46.62,123.19
Y6,149.81,29.21,72.83,223.38
Y7,174.23,21.32,111.80,232.21
Y8,186.47,36.17,98.79,277.43
Y9,140.23,25.35,62.79,214.24
Y10,33.84,5.82,18.44,50.75


## 2. 前処理の定義

各変数への前処理は次のとおり。学習が必要なもの（★）は学習データだけで fit する。

| 対象 | 前処理 | 作られる列 | 学習データ | テストデータ |
|---|---|---|---|---|
| 目的変数 | `feature_engineering.numeric.LogTransformer`（`TARGET_TRANSFORM="log"`） | `Yk`（変換後。元の値は `Yk_raw`） | ○ | （目的変数なし） |
| 目的変数 | `modeling.forecasting.TargetFeatureBuilder`（内部で `feature_engineering.time_series` のラグ・移動平均・変化率） | `Yk_lag_{n}`, `Yk_lag_1_ma_{w}`, `Yk_lag_1_roc_1` | ○ | **再帰予測の中で作る** |
| 日付 | `datetime_features.DatetimeFeaturesExtractor` | `date_year`, `date_month`, ... | ○ | ○ |
| 日付 | `datetime_features.CyclicalFeaturesEncoder`（月=12, 曜日=7） | `date_month_sin/cos`, `date_weekday_sin/cos` | ○ | ○ |
| 日付 | ★ `datetime_features.ElapsedTimeTransformer`（基準日 = 学習データの初日） | `date_elapsed_days` | ○ | ○ |
| 説明変数 | ★ `numeric.PandasPowerTransformer`（学習データで歪度が大きい列だけ） | 同じ列名で置き換え | ○ | ○ |
| 説明変数 | ★ `scaling.PandasRobustScaler` | 同じ列名で置き換え | ○ | ○ |

学習データで fit した前処理は `Preprocessor` にまとめ、テストデータには同じものの `transform` だけをかける。

In [5]:
class Preprocessor:
    """目的変数の変換・日付の特徴量・説明変数の変換をまとめた前処理（学習データだけで fit する）。

    目的変数のラグ・移動平均は含まない
    （学習データでは `TargetFeatureBuilder.build`、テスト期間では再帰予測で作る）。

    Attributes:
        target_transformer: 目的変数の変換（変換しないならNone）。全目的変数に同じ変換をかける。
        skewed: べき変換する説明変数（学習データの歪度で判定）。
        stateful: fit済みの、学習が必要な前処理（経過日数・べき変換・スケーリング）。
    """

    def fit(self, train: pd.DataFrame) -> "Preprocessor":
        """学習データで、変換の対象と値を決める。"""
        self.target_transformer = None
        if TARGET_TRANSFORM == "log":
            self.target_transformer = LogTransformer(TARGET_COLS, offset=TARGET_LOG_OFFSET).fit(
                train
            )
        elif TARGET_TRANSFORM is not None:
            raise ValueError(f"未対応の TARGET_TRANSFORM です: {TARGET_TRANSFORM}")
        dated = self._date_features(train)
        self.skewed = [c for c in feature_cols if abs(dated[c].skew()) > SKEW_THRESHOLD]
        steps: list[tuple[str, object]] = [
            ("elapsed", ElapsedTimeTransformer(DATE_COL, unit="days"))
        ]
        if self.skewed:
            steps.append(("power", PandasPowerTransformer(self.skewed, method="yeo-johnson")))
        steps.append(("scale", PandasRobustScaler(feature_cols)))
        self.stateful = Pipeline(steps).fit(dated)
        return self

    def transform(self, frame: pd.DataFrame) -> pd.DataFrame:
        """学習済みの値で変換する（各行の値だけから計算するので、行どうしは影響し合わない）。"""
        frame = frame.copy()
        targets = [t for t in TARGET_COLS if t in frame.columns]
        if targets:
            # 評価・出力用に元の値を残してから変換する
            raw = frame[targets].add_suffix("_raw")
            frame = pd.concat([frame, raw], axis=1)
            if self.target_transformer is not None:
                frame = self.target_transformer.transform(frame)
        return self.stateful.transform(self._date_features(frame))

    def inverse_target(self, target: str, values: np.ndarray) -> np.ndarray:
        """モデルの尺度の目的変数を、元の尺度に戻す。"""
        if self.target_transformer is None:
            return values
        # 変換は目的変数ごとに独立なので、1列だけを戻す
        frame = pd.DataFrame({t: values if t == target else np.nan for t in TARGET_COLS})
        return self.target_transformer.inverse_transform(frame)[target].to_numpy(dtype=float)

    @staticmethod
    def _date_features(frame: pd.DataFrame) -> pd.DataFrame:
        """日付の成分と周期特徴量を加える（各行の日付だけから作る。学習は不要）。"""
        frame = DatetimeFeaturesExtractor(DATE_COL, components=DATE_COMPONENTS).fit_transform(frame)
        if "month" in DATE_COMPONENTS:
            frame = CyclicalFeaturesEncoder(f"{DATE_COL}_month", period=12).fit_transform(frame)
        if "weekday" in DATE_COMPONENTS:
            frame = CyclicalFeaturesEncoder(f"{DATE_COL}_weekday", period=7).fit_transform(frame)
        return frame


def build_train_features(prep: Preprocessor, train: pd.DataFrame) -> pd.DataFrame:
    """学習データの前処理結果に、全目的変数のラグ・移動平均を加える（過去の実測値だけを使う）。

    各目的変数の特徴量は、その目的変数自身の過去の値だけから作る。
    ラグ・移動平均が作れない先頭の行（過去の日数が足りない行）は除く。
    """
    features = prep.transform(train)
    for builder in builders.values():
        features = builder.build(features)
    complete = features[all_target_feature_cols].notna().all(axis=1)
    return features[complete].reset_index(drop=True)

## 3. 前処理の実行と出力

In [6]:
prep = Preprocessor().fit(train_raw)
# 学習データ全体（先頭行も含む）の前処理結果。再帰予測の履歴として使う
train_prep = prep.transform(train_raw)
train_features = build_train_features(prep, train_raw)
test_features = prep.transform(test_raw)

date_feature_cols = [c for c in test_features.columns if c.startswith(f"{DATE_COL}_")]
# 目的変数ごとのモデルの特徴量（自分のラグ・移動平均 + 日付 + 説明変数）
model_feature_cols = {
    target: [*target_feature_cols[target], *date_feature_cols, *feature_cols]
    for target in TARGET_COLS
}
n_dropped = len(train_raw) - len(train_features)
print(
    f"学習データ: {len(train_features)}行（先頭 {n_dropped}行は目的変数のラグが作れないため除外）"
)
print(f"テストデータ: {len(test_features)}行")
print(f"モデルの特徴量（目的変数ごと）: {len(model_feature_cols[first])}列")
n_target_features = len(target_feature_cols[first])
print(
    f"  目的変数から（テストは再帰予測で作る）: {n_target_features}列 × {len(TARGET_COLS)}目的変数"
)
print(f"  日付から: {date_feature_cols}")
print(f"  説明変数: {len(feature_cols)}列（うちべき変換 {len(prep.skewed)}列: {prep.skewed}）")

# 出力（目的変数の列は先頭に置く。Yk・Yk_raw はモデルの特徴量に入れないこと）
target_cols_out = [c for t in TARGET_COLS for c in (t, f"{t}_raw")]
train_columns = [
    DATE_COL,
    *target_cols_out,
    *all_target_feature_cols,
    *date_feature_cols,
    *feature_cols,
]
test_columns = [DATE_COL, *date_feature_cols, *feature_cols]
for target, cols in model_feature_cols.items():
    assert not set(target_cols_out) & set(cols), f"{target}: 目的変数が特徴量に入っています"
    others = set(all_target_feature_cols) - set(target_feature_cols[target])
    assert not others & set(cols), f"{target}: 他の目的変数の特徴量が入っています"
assert not set(all_target_feature_cols) & set(test_columns), (
    "テストに目的変数の特徴量が入っています"
)
for filename, frame in (
    ("x50_train_features.csv", train_features[train_columns]),
    ("x50_test_features.csv", test_features[test_columns]),
):
    path = ensure_parent_dir(output_dir / filename)
    frame.to_csv(path, index=False, date_format="%Y-%m-%d")
    print(f"保存: {shown(path)}（{len(frame)}行 × {frame.shape[1]}列）")
test_features[test_columns].head()

学習データ: 1068行（先頭 28行は目的変数のラグが作れないため除外）
テストデータ: 365行
モデルの特徴量（目的変数ごと）: 69列
  目的変数から（テストは再帰予測で作る）: 9列 × 10目的変数
  日付から: ['date_year', 'date_month', 'date_day', 'date_weekday', 'date_week', 'date_month_sin', 'date_month_cos', 'date_weekday_sin', 'date_weekday_cos', 'date_elapsed_days']
  説明変数: 50列（うちべき変換 10列: ['X11', 'X12', 'X13', 'X14', 'X15', 'X16', 'X17', 'X18', 'X19', 'X20']）
保存: data/processed/x50/x50_train_features.csv（1068行 × 171列）
保存: data/processed/x50/x50_test_features.csv（365行 × 61列）


,date,date_year,date_month,date_day,date_weekday,date_week,date_month_sin,date_month_cos,date_weekday_sin,date_weekday_cos,date_elapsed_days,X1,X2,X3,X4,...,X36,X37,X38,X39,X40,X41,X42,X43,X44,X45,X46,X47,X48,X49,X50
0,2025-01-01,2025,1,1,3,1,0.5,0.866025,4.338837e-01,-0.900969,1096.0,0.612668,-0.090299,0.288626,-0.631648,...,0.50,-1.000000,0.000000,-0.25,0.0,0.745444,0.549043,1.151848,0.366719,-0.199045,-1.713255,-0.309753,-0.256991,0.202563,-0.365862
1,2025-01-02,2025,1,2,4,1,0.5,0.866025,-4.338837e-01,-0.900969,1097.0,0.749465,-0.225887,-0.841344,-0.700453,...,1.00,0.333333,0.666667,-0.75,0.0,-0.108289,-0.360833,0.022239,0.227996,0.007836,-0.125269,0.488382,0.638759,-0.516660,-0.848251
2,2025-01-03,2025,1,3,5,1,0.5,0.866025,-9.749279e-01,-0.222521,1098.0,1.201220,-0.642680,-0.067478,-0.772874,...,0.50,-0.666667,-0.333333,0.25,-0.5,0.795816,-0.279358,-0.333855,-0.021002,1.029903,-0.340940,0.266415,-0.288858,-2.462927,-0.351690
3,2025-01-04,2025,1,4,6,1,0.5,0.866025,-7.818315e-01,0.623490,1099.0,1.690621,-0.633936,-0.904482,-1.047453,...,-0.75,0.666667,1.000000,-0.75,-0.5,0.208369,-0.067860,0.593184,-0.465854,0.021760,-1.025799,0.760751,0.072671,-0.682169,0.783915
4,2025-01-05,2025,1,5,7,1,0.5,0.866025,-2.449294e-16,1.000000,1100.0,1.333267,-0.423032,-0.702015,-1.051530,...,-0.75,0.333333,-0.333333,-1.25,-1.0,-0.158573,0.344429,0.633601,-0.813775,0.557518,0.121524,-0.119419,-0.341279,-0.321372,0.091091


## 4. 再帰予測

目的変数ごとに、学習データ（その目的変数のラグ・移動平均を含む）で1日先を予測するモデルを学習する。
テスト期間は `modeling.forecasting.recursive_forecast` で1日ずつ進む。各日について:

1. 学習期間の実測値と、前日までの予測値をつないだ履歴から、その日の目的変数のラグ・移動平均・前日比を作る
   （`TargetFeatureBuilder`。学習データと同じ定義）。
2. その日の日付・説明変数の特徴量（`x50_test_features.csv` の値）と合わせて予測する。
3. 予測値を履歴に足し、翌日に進む。

予測値は対数の尺度なので、最後に元の尺度に戻す。

In [7]:
models: dict[str, LGBMRegressor] = {}
for target in TARGET_COLS:
    models[target] = LGBMRegressor(**MODEL_PARAMS, verbose=-1).fit(
        train_features[model_feature_cols[target]], train_features[target]
    )


def forecast(target: str, test_features: pd.DataFrame) -> np.ndarray:
    """テスト期間を再帰予測し、モデルの尺度（変換後）の予測値を返す。"""
    return recursive_forecast(
        models[target],
        train_prep,
        test_features,
        builders[target],
        model_feature_cols[target],
        apply_clip=False,
    )


prediction_parts = [test_features[[DATE_COL]]]
pred_model: dict[str, np.ndarray] = {}
for target in TARGET_COLS:
    pred_model[target] = forecast(target, test_features)
    pred = prep.inverse_target(target, pred_model[target])

    # 予測に使った目的変数の特徴量を、履歴（実測値 + 予測値）から作り直して記録する
    history = pd.concat(
        [
            train_prep[[DATE_COL, target]],
            test_features[[DATE_COL]].assign(**{target: pred_model[target]}),
        ],
        ignore_index=True,
    )
    recursion = builders[target].build(history).iloc[len(train_prep) :].reset_index(drop=True)
    # 作り直した特徴量で予測すると再帰予測の値と一致する（＝この特徴量を使って予測した）
    replay_input = pd.concat([recursion[target_feature_cols[target]], test_features], axis=1)
    replayed = models[target].predict(replay_input[model_feature_cols[target]])
    np.testing.assert_allclose(replayed, pred_model[target], rtol=1e-9, atol=1e-9)

    prediction_parts.append(
        pd.DataFrame({f"{target}_pred": pred, f"{target}_pred_model": pred_model[target]})
    )
    prediction_parts.append(recursion[target_feature_cols[target]])

predictions = pd.concat(prediction_parts, axis=1)
path = ensure_parent_dir(output_dir / "x50_test_predictions.csv")
predictions.to_csv(path, index=False, date_format="%Y-%m-%d")
print(f"保存: {shown(path)}（{len(predictions)}行 × {predictions.shape[1]}列）")
predictions[[DATE_COL, *[f"{t}_pred" for t in TARGET_COLS]]].head()

保存: data/processed/x50/x50_test_predictions.csv（365行 × 111列）


,date,Y1_pred,Y2_pred,Y3_pred,Y4_pred,Y5_pred,Y6_pred,Y7_pred,Y8_pred,Y9_pred,Y10_pred
0,2025-01-01,81.626048,67.140026,78.716079,33.962948,99.137765,161.723285,200.507647,207.742128,173.101385,28.209934
1,2025-01-02,87.266921,66.043190,69.574453,33.056948,88.182021,167.034857,209.969073,204.889992,156.073577,29.358314
2,2025-01-03,86.019992,66.841469,64.905437,31.203413,87.280023,167.465396,196.581867,200.645207,167.165586,34.912699
3,2025-01-04,87.635199,65.880720,66.954205,30.832434,76.177415,172.170405,202.481004,236.274983,158.262871,33.341451
4,2025-01-05,82.420733,68.917836,68.811331,33.479231,85.097788,193.234102,207.033081,230.369507,141.386773,34.776835


## 5. 予測の評価（正解ファイルがある場合）

正解はここで初めて読む（前処理・学習・予測には使っていない）。目的変数ごとに MAE を計算し、比較の基準として
学習データの最終日の値をそのまま使い続ける予測（ナイーブ予測）の MAE も示す。目的変数ごとに尺度が違うため、
MAE を平均の水準で割った値（相対MAE）も載せる。

In [8]:
if answer_path is not None:
    answer = read_csv_auto(answer_path)
    merged = predictions[[DATE_COL]].merge(answer, on=DATE_COL, how="left", validate="one_to_one")
    assert len(merged) == len(predictions) and merged[TARGET_COLS].notna().all().all()
    rows = []
    for target in TARGET_COLS:
        actual = merged[target]
        error = predictions[f"{target}_pred"] - actual
        naive_error = train_raw[target].iloc[-1] - actual
        rows.append(
            {
                "目的変数": target,
                "MAE（再帰予測）": error.abs().mean(),
                "MAE（ナイーブ）": naive_error.abs().mean(),
                "相対MAE（再帰予測）": error.abs().mean() / actual.mean(),
            }
        )
    evaluation = pd.DataFrame(rows)
    period = f"{merged[DATE_COL].min():%Y-%m-%d}〜{merged[DATE_COL].max():%Y-%m-%d}"
    print(f"評価: {len(merged)}日（{period}）")
    better = int((evaluation["MAE（再帰予測）"] < evaluation["MAE（ナイーブ）"]).sum())
    print(f"再帰予測がナイーブ予測より良い目的変数: {better} / {len(TARGET_COLS)}")
    display(evaluation.round(3))

評価: 365日（2025-01-01〜2025-12-31）
再帰予測がナイーブ予測より良い目的変数: 9 / 10


,目的変数,MAE（再帰予測）,MAE（ナイーブ）,相対MAE（再帰予測）
0,Y1,3.255,18.833,0.045
1,Y2,4.468,12.482,0.062
2,Y3,18.546,18.241,0.214
3,Y4,1.119,5.880,0.036
4,Y5,7.269,21.075,0.089
5,Y6,12.354,30.389,0.068
6,Y7,8.105,17.357,0.041
7,Y8,9.795,59.783,0.049
8,Y9,9.770,18.380,0.058
9,Y10,1.511,7.243,0.046


## 6. リークの確認

入力を書き換えて再実行し、未来の値や他の行・他の目的変数の値を使っていないことを確かめる。

1. **テストデータは学習側の結果に影響しない:** テストデータの説明変数を乱数に置き換えても、
   学習データの前処理結果は変わらない（前処理は学習データだけで fit している）。
2. **テストデータの前処理は行ごと:** テストデータのある1日の説明変数だけを書き換えると、変わるのはその日の行だけ。
3. **再帰予測は未来を見ない:** テスト期間のある日 t0 以降の説明変数を書き換えても、どの目的変数でも
   t0 より前の予測値は変わらない。
4. **学習データのラグは当日の値・他の目的変数を含まない:** 学習データのある日 t0 の Y1 だけを書き換えると、
   特徴量が変わるのは Y1 の特徴量の t0 の翌日以降の行だけ（Y2〜Y10 の特徴量は変わらない）。

In [9]:
rng = np.random.default_rng(0)


def with_random(frame: pd.DataFrame, mask: pd.Series, columns: list[str]) -> pd.DataFrame:
    """mask の行の指定列を、元とかけ離れた正の乱数に置き換えた表を返す。"""
    changed = frame.copy()
    changed.loc[mask, columns] = rng.uniform(1e3, 1e4, size=(int(mask.sum()), len(columns)))
    return changed


# 1. 前処理の fit にはテストデータを渡していないので、学習データの結果はテストデータに依存しない。
#    念のため、前処理をやり直しても学習データの結果が同じで、テストの変更が反映されることを確かめる
all_rows = pd.Series(True, index=test_raw.index)
changed_test = with_random(test_raw, all_rows, feature_cols)
pd.testing.assert_frame_equal(
    build_train_features(Preprocessor().fit(train_raw), train_raw), train_features
)
assert not prep.transform(changed_test).equals(test_features)
print("OK 1: テストデータは学習データの前処理結果に影響しない")

# 2. テストデータの1日だけを書き換えると、変わるのはその日の行だけ
t0 = test_raw[DATE_COL].iloc[100]
one_day = test_raw[DATE_COL] == t0
changed = prep.transform(with_random(test_raw, one_day, feature_cols))[test_columns]
same = changed.eq(test_features[test_columns]) | (
    changed.isna() & test_features[test_columns].isna()
)
diff_rows = ~same.all(axis=1)
assert diff_rows.equals(one_day), diff_rows[diff_rows].index.tolist()
print(f"OK 2: {t0:%Y-%m-%d} の説明変数を書き換えると、その日の行だけが変わる")

# 3. t0 以降の説明変数を書き換えても、t0 より前の予測値は同じ（全目的変数）
after = (test_raw[DATE_COL] >= t0).to_numpy()
changed_after = prep.transform(with_random(test_raw, pd.Series(after), feature_cols))
for target in TARGET_COLS:
    pred_changed = forecast(target, changed_after)
    np.testing.assert_array_equal(pred_changed[~after], pred_model[target][~after])
    assert not np.allclose(pred_changed[after], pred_model[target][after]), target
print(
    f"OK 3: {t0:%Y-%m-%d} 以降を書き換えても、全目的変数でそれより前の"
    f" {int((~after).sum())}日の予測値は同じ"
)

# 4. 学習データの t0 の Y1 だけを書き換えると、変わるのは Y1 の特徴量の翌日以降の行だけ
changed_target = TARGET_COLS[0]
t0_train = train_raw[DATE_COL].iloc[500]
changed_train = train_raw.copy()
changed_train.loc[changed_train[DATE_COL] == t0_train, changed_target] *= 10
rebuilt = build_train_features(prep, changed_train)
for target in TARGET_COLS:
    cols = target_feature_cols[target]
    diff = (rebuilt[cols] != train_features[cols]).any(axis=1)
    if target == changed_target:
        first_changed = rebuilt.loc[diff, DATE_COL].min()
        assert first_changed == t0_train + pd.Timedelta(days=1), first_changed
    else:
        assert not diff.any(), f"{target} の特徴量が {changed_target} の変更で変わりました"
print(
    f"OK 4: 学習データの {t0_train:%Y-%m-%d} の {changed_target} は、{changed_target} の"
    f"翌日 {first_changed:%Y-%m-%d} 以降の特徴量にだけ使われる（他の目的変数の特徴量は不変）"
)

OK 1: テストデータは学習データの前処理結果に影響しない
OK 2: 2025-04-11 の説明変数を書き換えると、その日の行だけが変わる


OK 3: 2025-04-11 以降を書き換えても、全目的変数でそれより前の 100日の予測値は同じ
OK 4: 学習データの 2023-05-16 の Y1 は、Y1 の翌日 2023-05-17 以降の特徴量にだけ使われる（他の目的変数の特徴量は不変）


## まとめ

- 学習データ・テストデータのそれぞれに、`src/feature_engineering` の前処理（目的変数の対数変換、日付の成分・周期・経過日数、
  説明変数のべき変換・スケーリング）をかけ、`data/processed/x50/` に CSV で出力した。前処理は学習データだけで fit した。
- 目的変数 Y1〜Y10 のラグ・移動平均・前日比は、学習データでは各目的変数自身の過去の実測値から作り、テスト期間では
  目的変数ごとの再帰予測の中で「実測値 + それまでの予測値」から1日ずつ作った
  （`x50_test_predictions.csv` に、予測に使った値を記録した）。
- 「リークの確認」で、テストデータ・未来の値・当日の目的変数・他の目的変数が前処理や予測に使われていないことを確かめた。

**注意点**

- 再帰予測では、先の日ほど「予測値から作ったラグ」に頼るため、誤差が積み重なる。
- 目的変数ごとに尺度が違うため、MAE は目的変数ごとに比べる（全体を比べるときは相対MAE を使う）。
- 日付の特徴量 `date_year` は、テスト期間（2025年）の値が学習データに無い。木のモデルは学習範囲の外の値を
  学習範囲の端と同じに扱う。トレンドのある目的変数（ダミーでは Y3・Y6・Y9）は、学習範囲より先の水準を
  当てにくい。
- 説明変数の欠損（X21〜X30）は埋めていない（LightGBM は欠損をそのまま扱える）。